# Task 3: Primary or Metastatic Inference

This notebook is a public release template for local evaluation.

Private images, labels, and task-specific fine-tuned checkpoints are not distributed in this repository. Update the configuration cell below with your own local paths before running the notebook.


In [ ]:
import os
import pathlib

def resolve_repo_root():
    cwd = pathlib.Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / 'finetune').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root containing finetune/.')

REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)
print(f'Repository root: {REPO_ROOT}')

CUDA_VISIBLE_DEVICES = '0'
RUN_FOLDS = [0, 1, 2, 3, 4]

BACKBONE_CHECKPOINT = REPO_ROOT / 'finetune' / 'checkpoints' / 'BoneFM.pth'
LABEL_CSV_TEMPLATE = '/path/to/private/primary_metastatic/fold_{fold}.csv'
MAIN_CHECKPOINT_TEMPLATE = '/path/to/private/task3_checkpoints/fold_{fold}.pth'
RELATIVE_CHECKPOINT_TEMPLATES = {
    "osteolytic": "/path/to/private/task3_checkpoints/fold_{fold}_extra_task_0.pth",
    "osteoblastic": "/path/to/private/task3_checkpoints/fold_{fold}_extra_task_1.pth",
    "type_of_primary_tumor": "/path/to/private/task3_checkpoints/fold_{fold}_extra_task_2.pth",
    "pathological_fracture": "/path/to/private/task3_checkpoints/fold_{fold}_extra_task_3.pth",
    "spinal_cord_compression": "/path/to/private/task3_checkpoints/fold_{fold}_extra_task_4.pth",
}
OUTPUT_DIR_TEMPLATE = 'finetune_logs/primary_metastatic_inference_fold_{fold}'


## Validate Private Assets

Run this cell after replacing the placeholder paths.

In [ ]:
from pathlib import Path

def validate_private_assets():
    missing = []
    if not BACKBONE_CHECKPOINT.exists():
        missing.append(f'backbone checkpoint: {BACKBONE_CHECKPOINT}')
    for fold in RUN_FOLDS:
        label_csv = Path(LABEL_CSV_TEMPLATE.format(fold=fold))
        main_ckpt = Path(MAIN_CHECKPOINT_TEMPLATE.format(fold=fold))
        if not label_csv.exists():
            missing.append(f'fold {fold} label csv: {label_csv}')
        if not main_ckpt.exists():
            missing.append(f'fold {fold} main checkpoint: {main_ckpt}')
    for task_name, template in RELATIVE_CHECKPOINT_TEMPLATES.items():
        ckpt_path = Path(template.format(fold=fold))
        if not ckpt_path.exists():
            missing.append(f"fold {fold} relative checkpoint for {task_name}: {ckpt_path}")
    if missing:
        raise FileNotFoundError('Missing required private assets\n' + '\n'.join(missing))
    print('All requested private assets are available.')

validate_private_assets()


## Run Inference

The cell below uses the public code with your private fold annotations and checkpoints.

In [ ]:
import argparse
from omegaconf import OmegaConf

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES

import finetune.trainer as trainer

def run_primary_metastatic_inference(fold_num):
    parser = argparse.ArgumentParser(description='BoneFM/BoneCoT inference')
    parser.add_argument('--config-file', type=str, default=str(REPO_ROOT / 'finetune' / 'configs' / 'task3_primary_or_metastatic_eval.yaml'))
    parser.add_argument('--output-dir', type=str, default=OUTPUT_DIR_TEMPLATE.format(fold=fold_num))
    parser.add_argument('--log-interval', type=int, default=10)
    parser.add_argument('--log_display', action='store_true', default=True)
    args = parser.parse_args([])

    default_cfg = OmegaConf.load(REPO_ROOT / 'finetune' / 'configs' / 'default_configs.yaml')
    user_cfg = OmegaConf.load(args.config_file)
    merged_cfg = OmegaConf.merge(default_cfg, user_cfg)
    for key, value in merged_cfg.items():
        key_str = str(key)
        if not hasattr(args, key_str) or getattr(args, key_str) is None:
            setattr(args, key_str, value)

    args.output_dir = os.path.abspath(args.output_dir)
    os.makedirs(args.output_dir, exist_ok=True)
    label_csv = Path(LABEL_CSV_TEMPLATE.format(fold=fold_num))
    args.data.train_dataset = str(label_csv)
    args.data.val_dataset = str(label_csv)
    args.data.test_dataset = str(label_csv)
    args.model.backbone_ckpt_path = str(BACKBONE_CHECKPOINT)
    args.trainer = 'BoneCoT_Eval_Trainer'
    args.model.main_task = 'primary_metastatic'
    main_ckpt = str(Path(MAIN_CHECKPOINT_TEMPLATE.format(fold=fold_num)))
    args.model.classifier_ckpt_path = main_ckpt
    if hasattr(args.model, 'model_ckpt_dict'):
        args.model.model_ckpt_dict[args.model.main_task] = main_ckpt
    for task_name, template in RELATIVE_CHECKPOINT_TEMPLATES.items():
        args.model.model_ckpt_dict[task_name] = str(Path(template.format(fold=fold_num)))
    task_trainer = getattr(trainer, args.trainer)(args)
    task_trainer.run()

for fold in RUN_FOLDS:
    print(f'Running fold {fold} for Task 3: Primary or Metastatic Inference...')
    run_primary_metastatic_inference(fold)
